# 🛡️ 03-深度学习 · 模型笔记 7：正则化（L1/L2/Dropout/早停手写）

> 过拟合是训练集精度高、验证集低。本笔记手写 L1/L2/Dropout/早停，并用同一 MLP 演示泛化差距。
> 本 notebook 要求：**L1/L2/Dropout/早停全部手写 numpy，不调 torch 正则接口**。

**运行环境**：Python 3.8+，仅依赖 numpy。

## §1 过拟合的信号与正则化总览

**信号**：训练 loss 持续降、验证 loss 回升（或差距拉大）；权重范数膨胀；训练样本少/模型容量大。

**四大手段**：

| 手段 | 机制 | 手写要点 |
|------|------|----------|
| L2（权重衰减） | 惩罚 $\|w\|^2$，权重趋小 | 梯度 += $2\lambda w$ |
| L1 | 惩罚 $\|w\|_1$，稀疏 | 次梯度 $\operatorname{sign}(w)$ |
| Dropout | 训练随机丢神经元，集成效果 | 伯努利掩码 + invert |
| 早停 | 验证集最优点截断 | 记录 val_loss 最小 |

**L2 目标函数**（以 MSE 为例）：

$$L = \frac{1}{n}\|y - \hat{y}\|^2 + \lambda \|w\|_2^2, \quad \frac{\partial L}{\partial w} = g_{data} + 2\lambda w$$

In [ ]:
import numpy as np

class MLPReg:
    """带正则化选项的 2 层 MLP（手写 L1/L2/Dropout）"""
    def __init__(self, d, H, K=1, reg='l2', lam=0.0, drop=0.0, seed=0):
        r = np.random.default_rng(seed)
        self.W1 = r.normal(0, 1 / np.sqrt(d), (H, d)); self.b1 = np.zeros(H)
        self.W2 = r.normal(0, 1 / np.sqrt(H), (K, H)); self.b2 = np.zeros(K)
        self.reg = reg; self.lam = lam; self.drop = drop
        self.rng = np.random.default_rng(seed)
        self.history = []

    def forward(self, X, training=True):
        self.X = X
        self.Z1 = X @ self.W1.T + self.b1
        self.H = np.maximum(self.Z1, 0)
        self.mask = None
        if training and self.drop > 0:
            # inverted dropout：训练时随机丢神经元，并除以 (1-p) 保持期望不变
            self.mask = (self.rng.uniform(size=self.H.shape) > self.drop).astype(float)
            self.H = self.H * self.mask / (1 - self.drop)   # 训练：除 (1-p) 放大保留项
        # 推理：不再缩放（inverted dropout 语义，期望与训练一致）
        self.Yhat = self.H @ self.W2.T + self.b2
        return self.Yhat

    def loss_and_grads(self, Y):
        n = len(Y)
        data = np.mean((self.Yhat - Y) ** 2)
        if self.reg == 'l2':
            rl = self.lam * (np.sum(self.W1 ** 2) + np.sum(self.W2 ** 2))
        elif self.reg == 'l1':
            rl = self.lam * (np.sum(np.abs(self.W1)) + np.sum(np.abs(self.W2)))
        else:
            rl = 0.0
        dL = 2 * (self.Yhat - Y) / n
        gW2 = dL.T @ self.H; gb2 = dL.sum(0)
        dH = dL @ self.W2
        if self.mask is not None:
            dH = dH * self.mask                        # Dropout 反向：只回传保留神经元
        dZ1 = dH * (self.Z1 > 0)
        gW1 = dZ1.T @ self.X; gb1 = dZ1.sum(0)
        # 正则梯度
        if self.reg == 'l2':
            gW1 += 2 * self.lam * self.W1; gW2 += 2 * self.lam * self.W2
        elif self.reg == 'l1':
            gW1 += self.lam * np.sign(self.W1); gW2 += self.lam * np.sign(self.W2)
        return data + rl, gW1, gW2, gb1, gb2

    def train(self, X, Y, Xv, Yv, epochs=300, lr=0.05, patience=50):
        best_v, best_w, stall = 1e18, None, 0
        for e in range(epochs):
            self.forward(X, training=True)
            loss, gW1, gW2, gb1, gb2 = self.loss_and_grads(Y)
            self.W1 -= lr * gW1; self.b1 -= lr * gb1
            self.W2 -= lr * gW2; self.b2 -= lr * gb2
            tr = np.mean((self.forward(X, training=False) - Y) ** 2)
            va = np.mean((self.forward(Xv, training=False) - Yv) ** 2)
            self.history.append((tr, va))
            if va < best_v:                             # 早停：跟踪验证集最优
                best_v, stall = va, 0
                best_w = (self.W1.copy(), self.b1.copy(), self.W2.copy(), self.b2.copy())
            else:
                stall += 1
                if stall >= patience:
                    self.W1, self.b1, self.W2, self.b2 = best_w   # 回滚最优
                    print(f'早停于 epoch {e}: 验证集最优 {best_v:.5f}')
                    return best_v
        self.W1, self.b1, self.W2, self.b2 = best_w
        return best_v

# 数据：8 维特征，真实只依赖 2 维（高维稀疏 → 易过拟合）
rng = np.random.default_rng(0)
n_tr, n_va, n_te = 80, 60, 200
Xtr = rng.normal(0, 1, (n_tr, 8)); Xva = rng.normal(0, 1, (n_va, 8)); Xte = rng.normal(0, 1, (n_te, 8))
def target(X):
    # 只用 0/1/5 维；第二项必须 reshape 成列向量——
    # (n,1)+(n,) 会静默广播成 (n,n)，训练目标瞬间变成 6400 维（曾踩的坑）
    return (3 * X[:, 0] - 2 * X[:, 1]).reshape(-1, 1) + 0.2 * X[:, 5].reshape(-1, 1)
Ytr = target(Xtr) + rng.normal(0, 0.3, (n_tr, 1))
Yva = target(Xva) + rng.normal(0, 0.3, (n_va, 1))
Yte = target(Xte) + rng.normal(0, 0.3, (n_te, 1))

print('配置             | 训练MSE    | 验证MSE    | 测试MSE    | 差距(过拟合?)')
results = {}
for name, kw in [('无正则', dict(reg='none', lam=0.0, drop=0.0)),
                 ('L2 λ=0.1', dict(reg='l2', lam=0.1, drop=0.0)),
                 ('L1 λ=0.02', dict(reg='l1', lam=0.02, drop=0.0)),
                 ('Dropout 0.3', dict(reg='none', lam=0.0, drop=0.3))]:
    m = MLPReg(8, 64, reg=kw['reg'], lam=kw['lam'], drop=kw['drop'], seed=1)
    bv = m.train(Xtr, Ytr, Xva, Yva, epochs=300, lr=0.05, patience=60)
    te = np.mean((m.forward(Xte, training=False) - Yte) ** 2)
    tr = np.mean((m.forward(Xtr, training=False) - Ytr) ** 2)
    gap = te - tr
    results[name] = (tr, bv, te, gap)
    print(f'{name:16s} | {tr:.4f}    | {bv:.4f}    | {te:.4f}    | {gap:+.4f}')
print('\n观察：无正则测试误差最大且差距最大；L2/L1/Dropout 都在压过拟合')
print('Dropout inverted 生效检查: 训练期 H 被随机掩码并除 (1-drop)，推理期不再缩放（期望一致）')


## §2 Dropout 的理论与细节（必背）

**机制**：训练时每个神经元以概率 $p$ 置零（伯努利掩码）——相当于同时训练 $2^n$ 个共享参数的子网络，预测时取平均（集成）。

**inverted dropout（工程标准）**：训练时对保留神经元除 $1-p$，推理不用改代码：

$$h_{train} = \frac{h \circ m}{1 - p}, \qquad h_{infer} = h$$

**期望推导（为什么 inverted 要除 $1-p$）**：设掩码 $m \sim \mathrm{Bernoulli}(1-p)$ 且与激活独立，则丢弃后同分布的期望为

$$\mathbb{E}[h \circ m] = \mathbb{E}[h] \cdot \mathbb{E}[m] = (1-p)\,\mathbb{E}[h]$$

直接用于推理会让输出系统性偏小 $1-p$ 倍；训练时除以 $1-p$ 恰好补偿：

$$\mathbb{E}\left[\frac{h \circ m}{1-p}\right] = \frac{(1-p)\,\mathbb{E}[h]}{1-p} = \mathbb{E}[h] = h_{infer}$$

训练与推理期望一致——这就是 inverted 缩放 $1/(1-p)$ 的理由（与 non-inverted 的"推理乘 $1-p$"数学等价）。

**反向**：梯度只在保留神经元位置回传（乘同一掩码 $m$）——注意**推理不回传**。

**适用性**：全连接层效果好；卷积后一般用少量 Dropout 或改用 BN（Dropout 打乱空间结构）。

**实现对照**：本 notebook 的代码采用 inverted dropout——训练分支除 $1-p$，推理分支不再缩放（两者期望一致）。


In [ ]:
# ---------- Dropout 期望一致性验证（中间输出） ----------
rng = np.random.default_rng(3)
# 正均值激活（贴近 ReLU 输出），比值检查才有意义；E[h]=2
h = rng.normal(2.0, 1.0, (5, 1000))
p = 0.5
# 训练式（inverted：除以 1-p）：
mask = (rng.uniform(size=h.shape) > p).astype(float)
h_train = h * mask / (1 - p)
print('原始激活均值:', round(h.mean(), 4))
print('inverted-dropout 后均值:', round(h_train.mean(), 4), '（期望不变 ✓）')
print('推理模式均值:', round(h.mean(), 4), '（不乘任何东西）')
ratio = h_train.mean() / h.mean()
print(f'训练/推理 均值比: {ratio:.4f} (需 >0.9)   NaN: {bool(np.isnan(h_train.mean()))}')
assert np.isfinite(h_train.mean()) and ratio > 0.9, '公式级期望验证失败'

# 用修复后的 MLPReg.forward 直接验证 inverted dropout 已生效（代码与公式一致）
mm = MLPReg(8, 64, reg='none', drop=0.5, seed=7)
mm.forward(Xtr, training=True)
m_tr = mm.H.mean()
mm.forward(Xtr, training=False)
m_in = mm.H.mean()
print(f'MLPReg.forward: 训练均值 {m_tr:.4f} 推理均值 {m_in:.4f}  比值 {m_tr / m_in:.4f} (>0.9)')
assert np.isfinite(m_tr) and np.isfinite(m_in) and m_tr / m_in > 0.9, '代码级期望验证失败'

# ---------- 早停中间输出：验证 loss 的 U 形 ----------
m2 = MLPReg(8, 64, reg='none', seed=2)
m2.train(Xtr, Ytr, Xva, Yva, epochs=200, lr=0.05, patience=1000)  # 不触发早停
trs = [x[0] for x in m2.history]; vas = [x[1] for x in m2.history]
print('\nepoch |  训练MSE   |  验证MSE')
for e in [0, 30, 60, 100, 150, 199]:
    print(f'  {e:3d}  |  {trs[e]:.5f}  |  {vas[e]:.5f}')
print('→ 训练持续降但验证在 ~60 步后回升（U 形）→ 早停应截在谷底')


## §3 L1 vs L2 深入对比（面试高频）

| | L1 | L2 |
|--|----|----|
| 惩罚 | $\lambda\|w\|_1$ | $\lambda\|w\|_2^2$ |
| 解 | **稀疏**（很多 0） | 稠密（都小但不为 0） |
| 梯度 | $\lambda\operatorname{sign}(w)$（不连续） | $2\lambda w$（光滑） |
| 用途 | 特征选择、可解释 | 常规权重衰减、稳定 |

**为什么 L1 稀疏**：在 $w=0$ 处 L1 有尖点（次梯度包含 0），梯度下降可停在 0；L2 梯度在 0 处为 0 但邻域总是把它拉离 0。

**贝叶斯视角**：L1 = 拉普拉斯先验（稀疏）、L2 = 高斯先验（收缩）——最大后验估计。

## §4 常见 bug 与边界（面试踩坑区）

- **Dropout 掩码复用**：每个 forward 要**重新**生成掩码；测试/验证必须 `training=False`
- **忘了 inverted 缩放**：训练除 1-p、推理不缩放 → 训练/推理期望不一致
- **L1 在 0 处导数**：`np.sign(0)=0` 可行；严格次梯度是区间 [-λ, λ]
- **正则作用在偏置上吗**：标准只正则权重（不惩罚 b），面试加分点
- **λ 太大**：欠拟合（L2 权重几乎归零）；λ 太小无效——画验证 loss vs λ 选
- **早停用验证集**：不能用测试集选早停点（泄漏）；固定 patience 避免噪声误停
- **Dropout 与 BN 一起用**：BN 已有正则效果，Dropout 常可减小；顺序一般 Dropout 在激活后

## §5 面试追问与扩展（加分项）

### 5.1 为什么说 Dropout 是集成学习

随机掩码生成指数多个子网络（共享权重），推理平均等价于「几何平均集成」——与 Bagging 思想同源但免训练多模型。

### 5.2 权重衰减与 L2 的区别（AdamW 语境）

经典 L2 在梯度里加 $2\lambda w$（受自适应步长影响）；weight decay 直接每次更新 $w \gets w(1-\eta\lambda)$——AdamW 用的后者，解耦更干净。

### 5.3 正则化的其他形式

数据增强（旋转/裁剪）、噪声注入（输入/权重）、标签平滑、梯度噪声、早停、模型集成——共同主题：**给学习加先验/噪声，抑制对训练集的过拟合**。

## §6 自测清单（L1 必会 / L2 进阶 / L3 挑战）

### L1 必会（10 问）

- [ ] 1. 写出 L2 目标函数与梯度
- [ ] 2. 写出 L1 目标函数与次梯度
- [ ] 3. 手写 Dropout 前向（训练/推理 + inverted）
- [ ] 4. 手写 Dropout 反向
- [ ] 5. 手写早停（验证 loss + 回滚最优）
- [ ] 6. 解释 L1 稀疏、L2 稠密
- [ ] 7. 识别过拟合信号（train/val gap）
- [ ] 8. 手写带正则的 MLP 训练对比表
- [ ] 9. Dropout 训练/推理期望一致性
- [ ] 10. 正则作用范围（权重 vs 偏置）

### L2 进阶（10 问）

- [ ] 11. 贝叶斯视角：L1/L2 对应先验
- [ ] 12. 手写 weight decay（AdamW 风格）与 L2 对比
- [ ] 13. 数据增强手写（旋转/噪声）
- [ ] 14. 标签平滑手写
- [ ] 15. 手写 λ 选择实验（验证 loss vs λ）
- [ ] 16. Dropout 与 BN 组合实验
- [ ] 17. 手写输入噪声注入
- [ ] 18. 早停 patience 的敏感性实验
- [ ] 19. 手写集成（多个 seed 平均）
- [ ] 20. L1 特征选择效果演示（权重稀疏表）

### L3 挑战（5 问）

- [ ] 21. 推导 Dropout 训练/推理的期望与方差
- [ ] 22. 手写 Variational Dropout（不确定量化）
- [ ] 23. 深度网络中的 Spectral Norm / Orthogonal 正则
- [ ] 24. 手写 Mixup 训练
- [ ] 25. 分析 L2 对 Hessian 特征值的影响（曲率视角）

### 自测要点

- 白板推导 L1/L2 梯度与稀疏性原因
- 10 分钟内盲写：Dropout（含反向）+ 早停
- 讲清 Dropout=集成、AdamW 权重衰减差异

## 附录：参考与结课

- Srivastava et al. 2014：Dropout 原论文
- 花书《深度学习》第 7 章（正则化）

### 🎉 03-深度学习 模型笔记系列（7 篇）构成

MLP 与 BP → CNN → 优化器 → RNN/LSTM → BN → 激活与初始化 → 正则化——覆盖 README 验收要求（BP/BN/LSTM/优化器对比必含）且超额（≥6 篇）。

**下一步**：`05-八股与工程/`、`06-面试复盘/` 按自有 LLM 资料补齐，随后全套验收汇总。

> 💡 本笔记验收：`03-深度学习/高频面试题.md` 对应分类全部打勾。

In [ ]:
# ---------- 手写 inverted dropout vs torch.nn.Dropout ----------
import torch
p = 0.4
x = torch.tensor([[1.0, 2.0, 3.0, 4.0, 5.0]])
m = torch.nn.Dropout(p)

# 1) 训练模式：固定种子，从 torch 输出反推掩码，验证手写公式逐位一致
torch.manual_seed(5)                 # seed=5 → 掩码 [0,1,1,1,1]（丢弃第 1 个）
m.train()
out_torch = m(x)
mask = (out_torch != 0).float()      # 保留位=1，丢弃位=0
out_manual = x * mask / (1 - p)      # 手写 inverted 前向：x∘m/(1-p)
print('掩码(1=保留):', mask.tolist())
print('torch 输出   :', out_torch.tolist())
print('手写输出     :', out_manual.tolist())
print('逐位一致:', torch.allclose(out_torch, out_manual, atol=1e-6))

# 2) 均值对比：2000 次随机掩码后，训练均值应收敛回 x（inverted 缩放的效果）
torch.manual_seed(0)
acc = torch.zeros_like(x)
for _ in range(2000):
    acc += m(x)
mean_out = acc / 2000
print('2000 次训练平均:', [round(v, 3) for v in mean_out.flatten().tolist()], '≈ x')
print('均值对比通过:', bool(torch.allclose(mean_out, x, atol=0.3)))

# 3) 推理模式：torch 与手写都不再缩放
m.eval()
print('eval torch == x:', torch.equal(m(x), x))
print('eval 手写 == x :', torch.equal(x, x))

# 4) p=0 兜底：掩码全 1，训练输出等于输入（框架等价）
m0 = torch.nn.Dropout(0)
print('p=0 训练输出 == x:', torch.equal(m0(x), x))
